<a href="https://colab.research.google.com/github/mhabibier/Practical-Statistics-for-Data-Scientists/blob/main/05_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mhabibier/Practical-Statistics-for-Data-Scientists/blob/main/05_Classification.ipynb)

# Chapter 5 — Classification

**Buku:** *Practical Statistics for Data Scientists: 50+ Essential Concepts Using R and Python*, edisi kedua (2020), Peter Bruce, Andrew Bruce, Peter Gedeck, O'Reilly Media.  
**Mahasiswa:** Muhammad Habibie Rabbani · TK-47-04 · Teknik Komputer, Telkom University.  
**Jenis pekerjaan:** reproduksi/adaptasi kode pendamping resmi, penjelasan teori berbahasa Indonesia, dan analisis tambahan.  
**Sumber kode:** [repository penulis pada commit tetap](https://github.com/gedeck/practical-statistics-for-data-scientists/tree/8a6d3bb6468e979c861d4b37215e1413702dfdfa); hak cipta kode sumber © 2019 Peter C. Bruce, Andrew Bruce, Peter Gedeck, GPL-3.0 (lihat `LICENSE` dan `THIRD_PARTY_NOTICES.md`).

Jalankan setiap sel dari atas ke bawah dalam sesi baru. Output disertakan sebagai contoh hasil aktual, bukan angka yang wajib diperoleh semua versi. Bagian berlabel **reproduksi eksploratif** mempelajari pola pada data buku; bagian **evaluasi prediktif** memakai data yang dipisahkan. Penjelasan dan kode disusun dengan bantuan LLM, diperiksa serta dieksekusi; mahasiswa perlu menelaah, menjalankan ulang, dan menuliskan refleksi sendiri.

## Tujuan dan peta pembahasan

Tujuan: menghitung probabilitas kelas, membedakan asumsi Naive Bayes/LDA/logistic regression, membaca koefisien dan diagnostik, mengevaluasi kelas langka, serta memilih strategi ketidakseimbangan tanpa kebocoran data.

| Bagian buku | Implementasi di notebook |
|---|---|
| Naive Bayes; fitur kategorikal dan numerik | One-hot + MultinomialNB seperti sumber; GaussianNB sebagai pelengkap |
| Discriminant analysis; covariance dan Fisher | LDA dua fitur dan garis batas dari koefisien model |
| Logistic regression, GLM, odds ratio, assessment | Logistic sklearn, GLM Binomial statsmodels, spline dan residual |
| Confusion matrix, rare class, ROC/AUC, lift | Evaluasi terpisah, PR/AP, lift dan kalibrasi tambahan |
| Undersampling, oversampling, weighting, synthetic data | Pipeline resampling training; SMOTE dan ADASYN hanya fitur kontinu |
| Cost-based classification; exploring predictions | Threshold dipilih pada validation, peta empat model dan LogisticGAM |

**Rancangan:** reproduksi ilustratif tetap memakai variabel buku. Benchmark prediktif membuang `status`, ID, dan `borrower_score` yang sudah dibentuk sebelumnya. Data test tidak menentukan scaler, encoder, hyperparameter, strategi resampling, maupun threshold.

## Persiapan lingkungan dan data

Python 3.12 digunakan saat pengujian. Sel berikut memasang pustaka tambahan hanya bila belum tersedia. Instalasi pertama memerlukan internet; jika lingkungan sudah siap, semua data tersedia dalam folder `data/` paket ZIP. Jika notebook dibuka sendiri di Colab, pembaca data mengunduh versi resmi yang dipatok pada commit tertentu dan memeriksa SHA-256. Tidak ada penggantian otomatis dengan data sintetis. `requirements-tested.txt` mencatat versi pengujian.

In [ ]:
import importlib.util, importlib.metadata, subprocess, sys
required = {'statsmodels': 'statsmodels>=0.14,<0.16', 'imblearn': 'imbalanced-learn>=0.13,<0.15', 'pygam': 'pygam>=0.10,<0.13'}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display
import sklearn
SEED = 42
rng = np.random.default_rng(SEED)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 110, "axes.titlesize": 11, "axes.labelsize": 10})
pd.set_option("display.max_columns", 24)
print("Python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__, "| sklearn", sklearn.__version__)
%matplotlib inline

from scipy.special import expit
from sklearn.base import clone
from sklearn.naive_bayes import MultinomialNB, GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
    precision_score, recall_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score, roc_curve, precision_recall_curve, brier_score_loss, log_loss)
from sklearn.calibration import calibration_curve
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
import statsmodels.api as sm
import statsmodels.formula.api as smf
from pygam import LogisticGAM, s

Python 3.13.16 | numpy 2.1.3 | pandas 2.2.3 | sklearn 1.6.1


In [ ]:
import urllib.request, hashlib
SOURCE_COMMIT = '8a6d3bb6468e979c861d4b37215e1413702dfdfa'
EXPECTED_SHA256 = {'loan200.csv': '20fa74aa858a19893d09f70e39d7fc1020dacfabead29a561136b07b554ba320', 'loan3000.csv': '64dcaf1c93d611a45276966bf9601201f47919136a7ecf375a60e74025a532f2', 'loan_data.csv.gz': 'de10007e08c98cbe563c88711fde73ccdc882474544a7e5c3d2d22a2cda6d7e1', 'full_train_set.csv.gz': '8e28117e1933b3ccda73b7415ceb438c951ff3cc7fd1c83a621e160d9c00a7eb', 'sp500_data.csv.gz': 'fb86ce1a70897800bee406a3c004254a731231041bb17aa4624e5fc3ab31cea7', 'sp500_sectors.csv': 'd4cfd1329b16aa5cbab7aeaaadbed2e21c0e78835d39aeb58abc58f1762cc337', 'housetasks.csv': '0adb7e7e50f387621e54ac572df5738da54ac66080b3b3f8b7c000d2c89393e7', 'sp500_18_stocks.csv.gz': '81f259c5909a75f96bbb3544739b88fec3c2125bc5867ce811c80582a3fd2150'}
DATA = Path("data")
DATA.mkdir(exist_ok=True)
STOCKS = sorted(["GOOGL", "AMZN", "AAPL", "MSFT", "CSCO", "INTC", "CVX", "XOM", "SLB", "COP", "JPM", "WFC", "USB", "AXP", "WMT", "TGT", "HD", "COST"])
def load_data(name, **kwargs):
    path = DATA / name
    if not path.exists():
        if name == "sp500_18_stocks.csv.gz":
            original = load_data("sp500_data.csv.gz", index_col=0)
            # Preserve all observations, select only the 18 symbols used in Chapter 7.
            return original.loc[:, STOCKS].copy()
        url = f"https://raw.githubusercontent.com/gedeck/practical-statistics-for-data-scientists/{SOURCE_COMMIT}/data/{name}"
        with urllib.request.urlopen(url, timeout=120) as response:
            payload = response.read()
        if hashlib.sha256(payload).hexdigest() != EXPECTED_SHA256[name]:
            raise ValueError(f"Checksum tidak cocok untuk {name}")
        path.write_bytes(payload)
    observed = hashlib.sha256(path.read_bytes()).hexdigest()
    if observed != EXPECTED_SHA256[name]:
        raise ValueError(f"Berkas lokal {name} berbeda dari versi yang diuji")
    return pd.read_csv(path, **kwargs)

def audit_frame(frame, name, target=None):
    print(f"{name}: {len(frame):,} baris × {frame.shape[1]} kolom")
    print("Missing:", int(frame.isna().sum().sum()), "| duplikasi semua kolom:", int(frame.duplicated().sum()))
    if target:
        display(frame[target].value_counts(dropna=False).rename("jumlah").to_frame().assign(proporsi=lambda x: x.jumlah / len(frame)))

## 1. Audit dataset dan definisi target

`loan_data` adalah sampel buku yang seimbang antara default dan paid off; `full_train_set` mempertahankan proporsi yang lebih timpang. Sampel yang diseimbangkan membantu demonstrasi, tetapi prior kelas dan probabilitasnya bukan otomatis probabilitas default populasi. `loan3000` adalah contoh kecil untuk geometri model.

Tetapkan **default = 1 (positif)** dan **paid off = 0**. `status` merupakan informasi hasil pinjaman, sehingga tidak boleh menjadi prediktor. `borrower_score` pada contoh pembentukan fitur sumber menggunakan label outcome; provenance terhadap split baru tidak terjamin. Oleh sebab itu, variabel tersebut hanya dipakai untuk reproduksi eksploratif, bukan skor generalisasi.

Audit missing/duplikasi tidak otomatis menghapus outlier. Nilai ekstrem keuangan bisa valid; menghapusnya dapat menghilangkan kasus penting. Dataset ini historis, sudah diproses penulis, dan bukan sampel representatif semua calon peminjam masa kini.

In [ ]:
loan = load_data("loan_data.csv.gz")
small = load_data("loan3000.csv")
full = load_data("full_train_set.csv.gz")
for name, frame in [("loan_data", loan), ("loan3000", small), ("full_train_set", full)]:
    audit_frame(frame, name, "outcome")
loan["default"] = loan.outcome.eq("default").astype(int)
small["default"] = small.outcome.eq("default").astype(int)
full["default"] = full.outcome.eq("default").astype(int)
NUM = ["payment_inc_ratio", "dti", "revol_bal", "revol_util"]
CAT = ["purpose_", "home_", "emp_len_"]
features = NUM + CAT
print("Duplikasi fitur+target benchmark:", loan[features + ["default"]].duplicated().sum())
display(loan[NUM].describe().T)
q1, q3 = loan[NUM].quantile(.25), loan[NUM].quantile(.75)
flags = (loan[NUM] < q1-1.5*(q3-q1)) | (loan[NUM] > q3+1.5*(q3-q1))
display(flags.sum().rename("flag_IQR_bukan_otomatis_salah").to_frame())

## 2. Naive Bayes — reproduksi eksploratif

Bayes menyatakan $P(Y=k\mid x)\propto P(Y=k)P(x\mid Y=k)$. Distribusi gabungan eksak sulit diestimasi ketika kombinasi fitur banyak dan sebagian tidak pernah muncul. Naive Bayes menyederhanakannya dengan asumsi **independensi kondisional**:

$$P(Y=k\mid x)\propto P(Y=k)\prod_j P(x_j\mid Y=k).$$

Kode buku menggunakan one-hot kategori dan `MultinomialNB`. Fitur menjadi indikator nonnegatif, sehingga dapat masuk model multinomial; indikator dalam satu kelompok kategori sendiri tidak independen, jadi probabilitasnya perlu dibaca hati-hati. `alpha` menghaluskan hitungan nol. `fit_prior=False` menetapkan prior kelas seragam. Smoothing sangat kecil dipertahankan untuk reproduksi, bukan rekomendasi universal.

Untuk fitur kontinu tersedia `GaussianNB`, yang mengasumsikan setiap fitur berdistribusi normal di dalam kelas, dengan kovariansi diagonal. Alternatifnya membagi nilai menjadi bin, dengan risiko kehilangan informasi. Fitur numerik tidak dilarang oleh seluruh keluarga Naive Bayes.

In [ ]:
nb_X = pd.get_dummies(loan[CAT], dtype=int)
nb_book = MultinomialNB(alpha=1e-10, fit_prior=False).fit(nb_X, loan.outcome)
row = nb_X.iloc[[146]]
display(pd.DataFrame(nb_book.predict_proba(row), columns=nb_book.classes_, index=["baris 146 (in-sample)"]))
print("Prediksi:", nb_book.predict(row)[0])
# Gaussian likelihood for continuous variables: illustration only, not a test score.
gnb = GaussianNB().fit(loan[NUM], loan.default)
display(pd.DataFrame(gnb.theta_, columns=NUM, index=["paid off", "default"]))

## 3. Discriminant analysis: covariance dan Fisher

LDA mengasumsikan $X\mid Y=k\sim N(\mu_k,\Sigma)$, dengan **matriks kovariansi bersama**. Elemen diagonal $\Sigma$ adalah varians; elemen lainnya menggambarkan perubahan bersama antarfitur. Berbeda dengan Naive Bayes Gaussian, korelasi fitur dapat dimodelkan.

Fungsi diskriminannya:

$$\delta_k(x)=x^T\Sigma^{-1}\mu_k-\tfrac12\mu_k^T\Sigma^{-1}\mu_k+\log\pi_k.$$

Untuk dua kelas, arah Fisher proporsional terhadap $\Sigma^{-1}(\mu_1-\mu_0)$: pemisahan rerata diperbesar relatif terhadap sebaran di dalam kelas. Garis keputusan muncul dari $\delta_1=\delta_0$. Gunakan `coef_` dan `intercept_` aktual; titik tengah rerata saja tidak selalu memasukkan prior kelas dengan benar. Pelanggaran normalitas atau perbedaan covariance antarkelas dapat membatasi LDA.

In [ ]:
two = ["borrower_score", "payment_inc_ratio"]
lda = LinearDiscriminantAnalysis(store_covariance=True).fit(small[two], small.default)
display(pd.DataFrame(lda.covariance_, index=two, columns=two))
display(pd.DataFrame(lda.means_, index=["paid off", "default"], columns=two))
xx = np.linspace(small.borrower_score.min(), small.borrower_score.max(), 150)
w = lda.coef_[0]; intercept = lda.intercept_[0]
fig, ax = plt.subplots(figsize=(7,4))
p = lda.predict_proba(small[two])[:, list(lda.classes_).index(1)]
sc = ax.scatter(small.borrower_score, small.payment_inc_ratio, c=p, cmap="coolwarm", s=12, alpha=.55)
ax.plot(xx, -(intercept + w[0]*xx)/w[1], color="black", label="P(default)=0,5")
ax.set(xlabel="borrower_score (fitur buku)", ylabel="payment_inc_ratio", ylim=(0,25), title="LDA: reproduksi eksploratif pada loan3000")
fig.colorbar(sc, ax=ax, label="Probabilitas default model")
ax.legend(); plt.tight_layout(); plt.show()

## 4. Logistic regression, logit dan GLM

Untuk respons biner, model memetakan kombinasi linear ke interval $(0,1)$:

$$\eta=\beta_0+\sum_j\beta_jx_j,\quad p=\frac{1}{1+e^{-\eta}},\quad \operatorname{logit}(p)=\log\frac{p}{1-p}=\eta.$$

Odds adalah $p/(1-p)$, berbeda dari probabilitas. Kenaikan satu unit fitur mengalikan odds dengan $e^{\beta_j}$ jika fitur lain tetap. Misalnya koefisien 0,5 menghasilkan odds ratio sekitar 1,65, bukan kenaikan probabilitas sebesar 65 poin persentase. Asosiasi ini tidak membuktikan sebab-akibat.

GLM terdiri atas distribusi respons, prediktor linear, dan link. Logistic regression menggunakan Bernoulli/Binomial dan logit; regresi linear biasanya menggunakan Gaussian dan link identitas. Logistic mengoptimalkan likelihood/log-loss, bukan residual sum of squares biasa. Residualnya tidak diasumsikan normal dengan varians tetap. Asumsi penting mencakup observasi yang memadai independensinya, bentuk logit yang sesuai, tidak adanya perfect separation, dan tidak adanya multikolinearitas sempurna.

In [ ]:
eta = np.linspace(-6,6,300)
fig, axes = plt.subplots(1,3,figsize=(12,3.4))
axes[0].plot(eta, expit(eta)); axes[0].set(xlabel="Prediktor linear eta", ylabel="p", title="Respons logistic")
pgrid = np.linspace(.01,.99,200)
axes[1].plot(pgrid,np.log(pgrid/(1-pgrid))); axes[1].set(xlabel="p",ylabel="logit(p)",title="Link logit")
axes[2].plot(eta,np.exp(eta)); axes[2].set(xlabel="Perubahan log-odds",ylabel="Odds ratio",yscale="log",title="Koefisien ke odds ratio")
plt.tight_layout(); plt.show()
book_X = pd.get_dummies(loan[["payment_inc_ratio", *CAT, "borrower_score"]], drop_first=True, dtype=float)
book_glm = sm.GLM(loan.default, sm.add_constant(book_X), family=sm.families.Binomial()).fit()
coef = pd.DataFrame({"beta":book_glm.params,"odds_ratio":np.exp(book_glm.params),"p_value":book_glm.pvalues})
ci = book_glm.conf_int()
coef["OR_CI_low"], coef["OR_CI_high"] = np.exp(ci[0]), np.exp(ci[1])
display(coef.round(5))
book_lr = LogisticRegression(C=1e6, max_iter=3000).fit(book_X, loan.default)
book_prob = book_lr.predict_proba(book_X)[:,1]
print("GLM converged:",book_glm.converged,"| AIC:",round(book_glm.aic,2),"| deviance:",round(book_glm.deviance,2))
print("Probabilitas 5 baris pertama:", np.round(book_prob[:5],4))
print("Log-probabilitas:", np.round(book_lr.predict_log_proba(book_X.iloc[:5]),4))

### 4.1 Menilai model dan bentuk nonlinear

P-value/interval koefisien mengukur ketidakpastian di bawah asumsi model; tidak menggantikan evaluasi pada data baru. AIC menggabungkan kecocokan dengan penalti jumlah parameter, $\mathrm{AIC}=-2\ell+2k$; bandingkan hanya model pada respons dan observasi yang sama. Deviance membandingkan likelihood dengan model jenuh.

Spline memungkinkan efek halus nonlinear pada skala logit. Kita memakai GLM **Binomial**, bukan regresi Gaussian terhadap label biner. Untuk diagnostik digunakan *working partial residual* $\eta_j+(y-p)/(p(1-p))$; ini berbeda dari residual mentah $y-p$. Nilainya dapat besar ketika probabilitas mendekati 0/1. Garis efek terpusat dan residual berikut bersifat deskriptif pada data yang sama, bukan bukti kemampuan generalisasi.

In [ ]:
spline_formula = "default ~ bs(payment_inc_ratio, df=8) + C(purpose_) + C(home_) + C(emp_len_) + bs(borrower_score, df=3)"
spline_glm = smf.glm(spline_formula, data=loan, family=sm.families.Binomial()).fit()
display(pd.DataFrame({"model":["logit linear","logit spline"],"AIC":[book_glm.aic,spline_glm.aic],"deviance":[book_glm.deviance,spline_glm.deviance]}))
names = spline_glm.model.exog_names
cols = [i for i,name in enumerate(names) if "bs(payment_inc_ratio" in name]
effect = spline_glm.model.exog[:,cols] @ spline_glm.params.iloc[cols].to_numpy()
effect = effect-effect.mean()
prob = np.clip(spline_glm.fittedvalues.to_numpy(),1e-8,1-1e-8)
working = (loan.default.to_numpy()-prob)/(prob*(1-prob))
idx = rng.choice(len(loan),size=2500,replace=False)
order = np.argsort(loan.payment_inc_ratio.to_numpy())
fig,ax=plt.subplots(figsize=(7,4))
ax.scatter(loan.payment_inc_ratio.iloc[idx],(working+effect)[idx],s=8,alpha=.12,label="Working partial residual (2.500 titik)")
ax.plot(loan.payment_inc_ratio.to_numpy()[order],effect[order],color="black",label="Efek spline terpusat")
ax.set(xlabel="payment_inc_ratio",ylabel="Skala logit / working residual",title="Diagnostik deskriptif GLM spline")
ax.legend(fontsize=8);plt.tight_layout();plt.show()

## 5. Evaluasi prediktif yang dipisahkan

Bagian ini menggunakan fitur dasar **tanpa borrower_score**. Data dibagi stratified 60% train, 20% validation, 20% test. Training CV memilih kekuatan regularisasi `C` menggunakan average precision (AP). Median imputation, standardisasi, dan kategori one-hot dipelajari hanya pada training fold. Validation memilih threshold biaya; test dibuka setelah pilihan dikunci.

Baris identik pada fitur+target dihapus sebelum split agar salinan persis tidak muncul di kedua sisi. Kesamaan fitur lintas split diaudit. Tanpa ID peminjam/waktu lengkap, pemisahan orang atau validasi masa depan belum dapat dijamin. Jika data produksi memiliki beberapa pinjaman per orang, gunakan grouped split; jika tujuan masa depan, gunakan split waktu.

Akurasi $=(TP+TN)/N$ dapat menipu untuk kelas langka. Precision $=TP/(TP+FP)$ mengukur ketepatan alarm, recall $=TP/(TP+FN)$ mengukur kasus positif yang ditemukan, specificity $=TN/(TN+FP)$ mengukur negatif yang benar. F1 menggabungkan precision dan recall. Semua label positif di sini adalah default.

In [ ]:
bench = loan[features+["default"]].drop_duplicates().copy()
X_train, X_hold, y_train, y_hold = train_test_split(bench[features],bench.default,test_size=.4,stratify=bench.default,random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_hold,y_hold,test_size=.5,stratify=y_hold,random_state=SEED)
for a,c in [(X_train,X_val),(X_train,X_test),(X_val,X_test)]:
    overlap = len(set(pd.util.hash_pandas_object(a,index=False)) & set(pd.util.hash_pandas_object(c,index=False)))
    assert overlap == 0, "Ada fitur identik lintas split; perlu grouped split"
display(pd.DataFrame({"split":["train","validation","test"],"n":[len(y_train),len(y_val),len(y_test)],"default_rate":[y_train.mean(),y_val.mean(),y_test.mean()]}))
def mixed_preprocessor():
    return ColumnTransformer([
        ("numeric",make_pipeline(SimpleImputer(strategy="median"),StandardScaler()),NUM),
        ("categorical",OneHotEncoder(handle_unknown="ignore",drop="first",sparse_output=False),CAT)])
pipe = Pipeline([("preprocess",mixed_preprocessor()),("model",LogisticRegression(max_iter=2000))])
cv = StratifiedKFold(n_splits=3,shuffle=True,random_state=SEED)
search = GridSearchCV(pipe,{"model__C":[.1,1.,10.]},scoring="average_precision",cv=cv,n_jobs=1)
search.fit(X_train,y_train)
display(pd.DataFrame(search.cv_results_)[["param_model__C","mean_test_score","std_test_score"]])
print("Pilihan berdasarkan training CV:",search.best_params_)
model = search.best_estimator_

### 5.1 ROC, precision–recall, lift, dan biaya

ROC memplot recall/TPR terhadap false positive rate $1-\mathrm{specificity}$. AUC menggambarkan kemampuan meranking pasangan positif di atas negatif; ia tidak memilih threshold dan tidak mengukur kalibrasi. Untuk kelas langka, precision–recall dan AP melengkapi ROC. AP bukan identik dengan luas trapezoidal kurva PR. Baseline AP kira-kira prevalensi kelas positif.

**Lift pada fraksi q** = precision dalam q bagian skor tertinggi dibagi prevalensi. Ini berguna jika pemeriksaan hanya dapat menangani sebagian kasus. Brier score adalah rata-rata $(p-y)^2$; semakin kecil semakin baik untuk probabilitas pada distribusi evaluasi tersebut.

Contoh biaya: satu false negative bernilai 5 unit, false positive 1 unit. Nilai tersebut hanya skenario pembelajaran. Untuk probabilitas terkalibrasi, biaya konstan, dan distribusi relevan, threshold teoretis $C_{FP}/(C_{FP}+C_{FN})$ berlaku. Di sini threshold praktis dipilih melalui validation, bukan test. Nilai pinjaman/manfaat berbeda-beda memerlukan expected utility per pinjaman, bukan satu threshold universal.

In [ ]:
def evaluate(y, p, threshold=.5):
    pred=(np.asarray(p)>=threshold).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    return {"accuracy":accuracy_score(y,pred),"balanced_accuracy":balanced_accuracy_score(y,pred),
        "precision_default":precision_score(y,pred,zero_division=0),"recall_default":recall_score(y,pred),
        "specificity":tn/(tn+fp),"F1":f1_score(y,pred),"ROC_AUC":roc_auc_score(y,p),
        "AP":average_precision_score(y,p),"Brier":brier_score_loss(y,p),"cost_per_case":(fp+5*fn)/len(y)}
val_p=model.predict_proba(X_val)[:,1]
thresholds=np.linspace(.05,.95,91)
validation_cost=pd.DataFrame([{**evaluate(y_val,val_p,t),"threshold":t} for t in thresholds])
chosen_threshold=float(validation_cost.loc[validation_cost.cost_per_case.idxmin(),"threshold"])
print("Threshold terkunci dari validation:",chosen_threshold)
fig,axes=plt.subplots(1,2,figsize=(11,3.5))
axes[0].plot(validation_cost.threshold,validation_cost.cost_per_case);axes[0].axvline(chosen_threshold,color="C3",ls="--")
axes[0].set(xlabel="Threshold",ylabel="Biaya rata-rata",title="Pemilihan pada validation")
fpr,tpr,_=roc_curve(y_val,val_p);axes[1].plot(fpr,tpr,label=f"AUC={roc_auc_score(y_val,val_p):.3f}")
axes[1].plot([0,1],[0,1],"--",color="gray");axes[1].set(xlabel="False positive rate",ylabel="Recall",title="ROC validation");axes[1].legend()
plt.tight_layout();plt.show()

In [ ]:
# Final test: no model or threshold changes after these results.
test_p=model.predict_proba(X_test)[:,1]
dummy=DummyClassifier(strategy="prior").fit(X_train,y_train)
dummy_p=dummy.predict_proba(X_test)[:,1]
final_results=pd.DataFrame({"Dummy prior":evaluate(y_test,dummy_p),"Logit threshold 0.5":evaluate(y_test,test_p),"Logit threshold validation":evaluate(y_test,test_p,chosen_threshold)}).T
display(final_results.round(4))
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
ConfusionMatrixDisplay.from_predictions(y_test,test_p>=chosen_threshold,labels=[0,1],display_labels=["paid off","default"],ax=axes[0],colorbar=False,cmap="Blues")
axes[0].grid(False)
axes[0].set_title("Test: threshold dari validation")
prec,rec,_=precision_recall_curve(y_test,test_p)
axes[1].plot(rec,prec,label=f"AP={average_precision_score(y_test,test_p):.3f}")
axes[1].axhline(y_test.mean(),ls="--",color="gray",label="Prevalensi test");axes[1].set(xlabel="Recall",ylabel="Precision",title="Precision–recall test");axes[1].legend(fontsize=8)
fraction,mean_p=calibration_curve(y_test,test_p,n_bins=8,strategy="quantile")
axes[2].plot(mean_p,fraction,"o-");axes[2].plot([0,1],[0,1],"--",color="gray");axes[2].set(xlabel="Rerata probabilitas model",ylabel="Proporsi default aktual",title="Kalibrasi pada sampel seimbang")
plt.tight_layout();plt.show()
order=np.argsort(-test_p)
lift_rows=[]
for q in [.1,.2,.3,.5,1.]:
    n=int(np.ceil(q*len(y_test)));positive=y_test.to_numpy()[order[:n]].mean()
    lift_rows.append({"fraksi_diperiksa":q,"precision_top":positive,"lift":positive/y_test.mean()})
display(pd.DataFrame(lift_rows).round(3))
print("Skor test berlaku untuk sampel buku; bukan probabilitas populasi calon peminjam.")

## 6. Strategi data tidak seimbang

Undersampling mengurangi mayoritas tetapi kehilangan informasi; random oversampling menggandakan minoritas tetapi tidak menambah informasi baru. Class weighting meningkatkan kontribusi kesalahan kelas tertentu pada loss. Ketiganya dapat mengubah keputusan dan kalibrasi.

SMOTE membuat $x_{new}=x_i+\lambda(x_{neighbor}-x_i)$ pada tetangga minoritas. ADASYN menambah lebih banyak sampel di daerah minoritas yang sulit menurut lingkungan mayoritas; hal ini juga dapat memperbesar noise/outlier. Interpolasi biasa pada one-hot dapat menghasilkan kategori pecahan yang tidak valid. Karena itu perbandingan berikut menggunakan empat **fitur kontinu yang sama** untuk seluruh metode. Untuk data campuran, pertimbangkan SMOTENC dan validasi domain, bukan memaksa SMOTE biasa pada semua kolom.

Sampel stratified maksimum 24.000 observasi dari full_train_set dipakai untuk membatasi waktu. Resampling hanya terjadi pada training melalui `imblearn.Pipeline`. Validation/test tetap memiliki proporsi asli sampel tersebut. Dataset ini beririsan dengan loan_data, sehingga eksperimen ini bukan replikasi independen untuk menggabungkan interval kepercayaan dari benchmark sebelumnya.

In [ ]:
imbal=full[NUM+["default"]].drop_duplicates().copy()
if len(imbal)>24000:
    imbal,_=train_test_split(imbal,train_size=24000,stratify=imbal.default,random_state=SEED)
iX, iH, iy, ih = train_test_split(imbal[NUM],imbal.default,test_size=.4,stratify=imbal.default,random_state=SEED)
vX, tX, vy, ty = train_test_split(iH,ih,test_size=.5,stratify=ih,random_state=SEED)
print("n train/val/test:",len(iy),len(vy),len(ty),"| default rate:",round(iy.mean(),4),round(vy.mean(),4),round(ty.mean(),4))
strategies={"Original":None,"Class weight balanced":"weight","Undersampling":RandomUnderSampler(random_state=SEED),"Oversampling":RandomOverSampler(random_state=SEED),"SMOTE":SMOTE(random_state=SEED),"ADASYN":ADASYN(random_state=SEED)}
imb_models={}; imb_rows=[]
for name,sampler in strategies.items():
    steps=[("impute",SimpleImputer(strategy="median")),("scale",StandardScaler())]
    if sampler is not None and not isinstance(sampler,str):steps.append(("resample",sampler))
    steps.append(("model",LogisticRegression(max_iter=2000,class_weight="balanced" if isinstance(sampler,str) else None)))
    m=ImbPipeline(steps).fit(iX,iy);imb_models[name]=m
    pr=m.predict_proba(vX)[:,1]
    imb_rows.append({"strategy":name,**evaluate(vy,pr)})
imb_validation=pd.DataFrame(imb_rows).set_index("strategy")
display(imb_validation.round(4))
imb_choice=imb_validation.AP.idxmax()
print("Strategi terpilih berdasarkan AP validation:",imb_choice)
# Locked strategy; default threshold 0.5 is predeclared for this experiment.
imb_test=evaluate(ty,imb_models[imb_choice].predict_proba(tX)[:,1])
display(pd.DataFrame([imb_test],index=["Final test: "+imb_choice]).round(4))

### Membaca perbandingan ketidakseimbangan

Recall yang naik bisa dibayar dengan lebih banyak false positive. Resampling tidak dijamin memperbaiki AP/AUC; bandingkan metrik sesuai tujuan. Strategi dengan AP terbaik dipilih sebelum test, sedangkan threshold 0,5 tetap untuk memperlihatkan dampak perubahan distribusi training. Untuk deployment, diperlukan validasi biaya dan kalibrasi pada prevalensi yang sesuai.

## 7. Exploring predictions: bentuk batas keputusan

LDA dan logistic dengan prediktor linear memiliki batas lurus; tree membuat wilayah berbentuk potongan sejajar sumbu; model aditif logistic memungkinkan efek halus nonlinear. Di sini **LogisticGAM** dipakai menggantikan `LinearGAM` pada sumber agar respons biner memiliki link logit dan probabilitas valid. Peta memakai data buku yang sama untuk ilustrasi, termasuk borrower_score. Ini tidak dipakai memilih model akhir benchmark.

In [ ]:
models={"LDA":LinearDiscriminantAnalysis(),"Logistic":make_pipeline(StandardScaler(),LogisticRegression(max_iter=2000)),"Decision tree":DecisionTreeClassifier(criterion="entropy",min_impurity_decrease=.003,random_state=SEED)}
for m in models.values():m.fit(small[two],small.default)
gam=LogisticGAM(s(0,n_splines=8)+s(1,n_splines=8),lam=10).fit(small[two].to_numpy(),small.default.to_numpy())
gx,gy=np.meshgrid(np.linspace(.15,.85,110),np.linspace(0,25,110))
grid=pd.DataFrame({two[0]:gx.ravel(),two[1]:gy.ravel()})
fig,axes=plt.subplots(2,2,figsize=(10,8))
for ax,(name,m) in zip(axes.flat,[*models.items(),("Logistic GAM",gam)]):
    prob=m.predict_mu(grid.to_numpy()) if name=="Logistic GAM" else m.predict_proba(grid)[:,1]
    z=prob.reshape(gx.shape)
    ax.contourf(gx,gy,z,levels=np.linspace(0,1,11),cmap="coolwarm",vmin=0,vmax=1)
    ax.contour(gx,gy,z,levels=[.5],colors="black",linewidths=1)
    ax.set(title=name,xlabel="borrower_score",ylabel="payment_inc_ratio")
fig.suptitle("Reproduksi eksploratif: warna menunjukkan P(default)")
plt.tight_layout();plt.show()

## Analisis hasil eksekusi yang disertakan

Hasil berikut berasal dari eksekusi pada 10 Oktober 2026 (WIB), dengan seed dan versi pada `requirements-tested.txt`.

- Naive Bayes pada contoh baris 146 menghasilkan P(default) sekitar **0,654**. Ini prediksi in-sample untuk reproduksi, bukan validasi kemampuan model.
- Pada GLM, koefisien payment_inc_ratio sekitar **0,0797**, sehingga odds default berubah sekitar **1,083 kali per satu unit**, dengan fitur lain tetap. Interpretasi ini bersyarat pada model dan dataset buku. Angka p-value yang ditampilkan 0 karena pembulatan bukan probabilitas tepat nol.
- AIC spline sekitar **57.504,28**, lebih kecil daripada logit linear **57.538,80** pada data sama. Peningkatan kecocokan terpenalti ini tidak langsung membuktikan skor test lebih baik.
- Benchmark tanpa borrower_score memperoleh **AUC test 0,6546**, **AP 0,6375**, dan akurasi **61,72%** pada threshold 0,5. Dummy memiliki AUC 0,5 dan AP sekitar 0,5. Kemampuan ranking di atas baseline masih cukup terbatas.
- Threshold biaya hasil validation adalah **0,23**. Recall test naik dari **60,63%** menjadi **99,65%**, tetapi specificity turun dari **62,80%** menjadi **1,65%**. Model hampir selalu memberikan alarm default. Biaya test sekitar **0,5006/kasus**, sedikit lebih buruk daripada baseline selalu-positif sekitar **0,5001/kasus**. Ini menunjukkan bahwa threshold terpilih di validation tidak dijamin mengungguli baseline pada test; tidak dilakukan pemilihan ulang berdasarkan test.
- Pada sampel tidak seimbang, metode Original memiliki AP validation tertinggi, tetapi pada threshold 0,5 recall test hanya **0,11%** meski akurasi **81,12%**. AP menilai ranking, bukan kualitas alarm pada satu threshold. Jadi strategi terpilih berdasarkan AP ini **belum memadai sebagai pendeteksi default** pada threshold tersebut; perancangan tujuan/threshold perlu dilakukan pada validation atau studi baru.

Hasil yang rendah tetap disajikan agar analisis tidak menyamakan 'kode benar' dengan 'akurasi harus sangat tinggi'.

## Ringkasan chapter

Naive Bayes menyederhanakan likelihood, LDA memodelkan distribusi kelas dengan covariance bersama, dan logistic regression memodelkan log-odds. Spline/GAM menambah keluwesan, tetapi interpretasi dan diagnostik tetap diperlukan. Evaluasi harus memisahkan training, validation, dan test; kelas positif serta threshold harus dinyatakan. ROC/AUC, PR/AP, confusion matrix, lift, dan kalibrasi menjawab pertanyaan berbeda. Resampling dan biaya kesalahan harus diuji sesuai tujuan, tanpa memberi data test peran dalam pemilihan model. Data buku yang diseimbangkan dan fitur hasil pembentukan sebelumnya membatasi interpretasi probabilitas/populasi.

## Refleksi mandiri

Jawab berdasarkan output dan pemahaman Anda; jangan sekadar menyalin rangkuman.

1. Mengapa status dan borrower_score tidak digunakan dalam benchmark test?
2. Bandingkan recall dan specificity pada threshold 0,5 dengan threshold hasil validation. Apa konsekuensi praktisnya?
3. Metode resampling mana yang terpilih? Apakah recall tertinggi juga memberikan AP tertinggi?
4. Mengapa odds ratio 2 tidak berarti probabilitas dua kali lipat untuk semua nilai awal?
5. Apa yang perlu berubah sebelum model boleh digunakan pada data pinjaman masa depan?

## Referensi

1. Bruce, P., Bruce, A., & Gedeck, P. (2020). *Practical Statistics for Data Scientists*, 2nd ed. O’Reilly Media.
2. [Kode pendamping resmi dan lisensi](https://github.com/gedeck/practical-statistics-for-data-scientists/tree/8a6d3bb6468e979c861d4b37215e1413702dfdfa). Cakupan diverifikasi melalui kode chapter resmi; PDF pada tautan tugas tidak dapat dibaca melalui akses yang tersedia.
3. [Daftar isi edisi kedua](https://www.oreilly.com/library/view/practical-statistics-for/9781492072935/).
4. [scikit-learn: Naive Bayes](https://scikit-learn.org/stable/modules/naive_bayes.html).
5. [scikit-learn: model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html).
6. [statsmodels: Generalized Linear Models](https://www.statsmodels.org/stable/glm.html).
7. [imbalanced-learn: common pitfalls](https://imbalanced-learn.org/stable/common_pitfalls.html).